In [70]:
import torch
import pickle
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import selfies as sf
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, DataStructs, rdmolops, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import math
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from sklearn.metrics import r2_score
from IPython.display import display
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch_geometric.loader import DataLoader
import seaborn as sns
import os
import re
torch.cuda.empty_cache()

In [12]:
df = pd.read_csv('/Users/mateusz/Documents/apki/Graph-Latent-Space/data/raw/smiles_selfies_full.csv')

In [13]:
df.head()

,smiles,selfies
0,O=S(O)c1cc2c(cc1F)OC(c1ccc(F)cc1F)(c1ccc(F)cc1...,[O][=S][Branch1][C][O][C][=C][C][=C][Branch1][...
1,CN(C)Cc1cccc(C2Nc3cccc4c(=O)[nH]nc(c34)C2c2ccc...,[C][N][Branch1][C][C][C][C][=C][C][=C][C][Bran...
2,O=C(N[C@@H](CO)c1nc2cc(Cl)ccc2[nH]1)c1ccc(C(=O...,[O][=C][Branch2][Ring1][#Branch1][N][C@@H1][Br...
3,O=C(Cn1cc(I)cn1)N1CCCc2c1cnn2-c1ccc(F)cc1,[O][=C][Branch1][N][C][N][C][=C][Branch1][C][I...
4,Cc1ccc(-c2ccnc(Cl)c2)n1CC(=O)OCc1ccccc1,[C][C][=C][C][=C][Branch1][N][C][=C][C][=N][C]...


In [44]:
# convert to graphs
SUPPORTED_ATOMS = [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]

def atom_to_feature_vector(atom):
    """zamienia atom na wektor cech, ten wektor będzie potem nodem w grafie"""
    atomic_num = atom.GetAtomicNum()
    # one hot encoding
    return [1 if atomic_num == atom_type else 0 for atom_type in SUPPORTED_ATOMS]

def smiles_to_graph(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return None
    
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append(atom_to_feature_vector(atom))
    x = torch.tensor(atom_features, dtype=torch.float)

    edges = []
    for bond in mol.GetBonds():
        edges.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
        edges.append((bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()))
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(x=x, edge_index=edge_index)
    return data
    

In [49]:
smiles = "CC(CC)O" 
graph = smiles_to_graph(smiles)
print(graph)
print(graph.x)
print(graph.edge_index)

Data(x=[5, 10], edge_index=[2, 8])
tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]])
tensor([[0, 1, 1, 2, 2, 3, 1, 4],
        [1, 0, 2, 1, 3, 2, 4, 1]])


In [ ]:
GRAPH_PATH = '/Users/mateusz/Documents/apki/Graph-Latent-Space/data/processed/graphs.pt'

if os.path.exists(GRAPH_PATH):
    graph_list = torch.load(GRAPH_PATH)
else:
    graph_list = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        smiles = row['smiles']
        graph = smiles_to_graph(smiles)
        graph_list.append(graph)
    torch.save(graph_list, GRAPH_PATH)

In [62]:
# split dataset
graph_train, graph_temp = train_test_split(graph_list, test_size=0.2, random_state=42, shuffle=True)
graph_val, graph_test = train_test_split(graph_temp, test_size=0.5, random_state=42, shuffle=True)

class GraphDataset(Dataset):
    def __init__(self, graph_list):
        self.graphs = graph_list

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        return self.graphs[index]

train_dataset = GraphDataset(graph_train)
val_dataset = GraphDataset(graph_val)
test_dataset = GraphDataset(graph_test)

In [74]:
# model
class GraphVAE(nn.Module):
    def __init__(self, latent_dim, max_nodes=37, in_channels=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.max_nodes = max_nodes
        self.in_channels = in_channels

        # Encoder
        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 128)

        # Latent space
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # Decoder
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
        )

        self.decoder_node = nn.Linear(256, max_nodes * in_channels)
        self.decoder_adj = nn.Linear(256, max_nodes * max_nodes)
    
    def encode(self, x, edge_index, batch):
        h = self.conv1(x, edge_index).relu()
        h = self.conv2(h, edge_index).relu()
        h_graph = global_mean_pool(h, batch)
        
        mu = self.fc_mu(h_graph)
        logvar = self.fc_logvar(h_graph)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu
    
    def decode(self, z):
        h = self.decoder_fc(z)
        
        recon_x = self.decoder_node(h).view(-1, self.max_nodes, self.in_channels)
        recon_adj = self.decoder_adj(h).view(-1, self.max_nodes, self.max_nodes)
        recon_adj = torch.sigmoid(recon_adj)
        
        return recon_x, recon_adj
    
    def forward(self, x, edge_index, batch):
        mu, logvar = self.encode(x, edge_index, batch)
        z = self.reparameterize(mu, logvar)
        recon_x, recon_adj = self.decode(z)

        # padding to max_nodes
        true_x, _ = to_dense_batch(x, batch, max_num_nodes=self.max_nodes)
        true_adj = to_dense_adj(edge_index, batch, max_num_nodes=self.max_nodes)

        return recon_x, recon_adj, true_x, true_adj, mu, logvar

def vae_graph_loss(recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=0.01, edge_weight=1.0):
    in_channels = true_x.size(-1)
    logits = recon_x.reshape(-1, in_channels)
    targets = true_x.argmax(dim=-1).reshape(-1)

    # loss for atoms
    atom_loss = F.cross_entropy(logits, targets)

    # loss for edges
    edge_loss = F.binary_cross_entropy(recon_adj.reshape(-1), true_adj.reshape(-1))

    recon_loss = atom_loss + edge_weight * edge_loss

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total_loss = recon_loss + beta * kl

    return total_loss, atom_loss.item(), edge_loss.item(), kl.item()



In [ ]:
# training
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

latent_dim = 64
max_nodes = 37
in_channels = 10
model = GraphVAE(latent_dim, max_nodes=max_nodes, in_channels=in_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

beta = 0.01
edge_weight = 1.0
num_epochs = 50
history = {'train_loss': [], 'atom_loss': [], 'edge_loss': [], 'kl_loss': [], 'val_loss': []}
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0
    total_atom_loss = 0
    total_edge_loss = 0
    total_kl_loss = 0
    pbar = tqdm(train_loader)
    for graph in pbar:
        x, edge_index, batch = graph.x.to(device), graph.edge_index.to(device), graph.batch.to(device)

        optimizer.zero_grad()

        recon_x, recon_adj, true_x, true_adj, mu, logvar = model(x, edge_index, batch)

        loss, atom_l, edge_l, kl_l = vae_graph_loss(
            recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=beta, edge_weight=edge_weight
        )
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix({
            "atom": f"{atom_l:.3f}",
            "edge": f"{edge_l:.3f}",
            "kl": f"{kl_l:.3f}", 
            "tot": f"{loss.item():.3f}",
        })

        total_loss += loss.item()
        total_atom_loss += atom_l
        total_edge_loss += edge_l
        total_kl_loss += kl_l
    
    with torch.no_grad():
        model.eval()
        val_loss = 0
        for graph in val_loader:
            x, edge_index, batch = graph.x.to(device), graph.edge_index.to(device), graph.batch.to(device)
            recon_x, recon_adj, true_x, true_adj, mu, logvar = model(x, edge_index, batch)

            loss, _, _, _ = vae_graph_loss(
                recon_x, recon_adj, true_x, true_adj, mu, logvar, beta=beta, edge_weight=edge_weight
            )
            val_loss += loss.item()
    
    total_loss /= len(train_loader)
    total_atom_loss /= len(train_loader)
    total_edge_loss /= len(train_loader)
    total_kl_loss /= len(train_loader)
    val_loss /= len(val_loader)
    history['train_loss'].append(total_loss)
    history['atom_loss'].append(total_atom_loss)
    history['edge_loss'].append(total_edge_loss)
    history['kl_loss'].append(total_kl_loss)
    history['val_loss'].append(val_loss)
    print(f"Epoch {epoch}/{num_epochs} - Train Loss: {total_loss:.4f} (Atom: {total_atom_loss:.4f}, Edge: {total_edge_loss:.4f}, KL: {total_kl_loss:.4f}) - Val Loss: {val_loss:.4f}")

MODEL_PATH = '/Users/mateusz/Documents/apki/Graph-Latent-Space/models/graph_vae.pt'
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model zapisany w: {MODEL_PATH}")

  0%|          | 0/19861 [00:00<?, ?it/s]

KeyboardInterrupt: 